# Project Hail Mary: The Adrian Descent
## Assignment 2 — Physics-Based Tabular Q-Learning
**Architecting a Physics-Based RL Environment for autonomous probe control in Adrian's atmosphere.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, time
from IPython.display import clear_output


---
## Part 1 — Rocky's Analytics (Pre-code Calculations)

### Terminal Velocity
When the spin-drive fails completely, at steady state:  
$$F_{gravity} = F_{drag} \implies m \cdot g = k_{drag} \cdot v_{term}^2$$
$$v_{term} = \sqrt{\frac{m \cdot g}{k_{drag}}}$$

### Maximum Net Acceleration (thrust ON, v=0)
$$a_{net} = \frac{F_{thrust} + F_{gravity}}{m} = \frac{25000 - 13700}{1000}$$


In [ ]:
# ── Rocky's Analytics ─────────────────────────────────────────────────────────
m, g, k_drag, F_thrust, R = 1000.0, 13.7, 2.0, 25_000.0, 10_700_000.0

v_term = np.sqrt(m * g / k_drag)
print(f"Terminal velocity (drive OFF): {v_term:.2f} m/s")
print(f"  → Safe catch limit is -3.0 m/s. {v_term:.1f} m/s far exceeds it.")
print(f"  → Without thrust the probe WILL be destroyed. Control is mandatory.\n")

h_drop = 1000.0
F_grav_at_drop = -m * g * (1 - h_drop / R)
a_max = (F_thrust + F_grav_at_drop) / m
print(f"Net acceleration (thrust ON, v=0, h=1000 m): {a_max:.4f} m/s²")
print(f"  → {a_max:.2f} m/s² upward: thrust CAN overcome gravity — but only barely.")
print(f"  → A lazy agent that keeps thrust ON will slowly drift upward (runaway probe).")


---
## Design Questions — Answers

### Q1 · Memory in the Storm (Transition Probability Matrix vs. random roll)
A random-die model assigns the next wind state **independently** of the current one — every step
has the same fixed probability of any wind level regardless of what just happened.  
A **Transition Probability Matrix** (TPM) encodes *momentum*: given that we are currently in a
Calm state, the next state is most likely still Calm (p=0.70), less likely Gusty (p=0.25), and
rarely an Adrian Gale (p=0.05). This matches real atmospheric physics where weather changes
gradually, making the resulting environment far more realistic and trainable because the agent
can learn short-horizon correlations between successive wind observations.

---
### Q2 · Exploration vs. Exploitation (ε after 5 000 episodes)


In [ ]:
eps = 1.0
for _ in range(5000):
    eps *= 0.999
print(f"ε after 5 000 episodes: {eps:.4f}  (~{eps*100:.2f}%)")
print("The agent is almost entirely EXPLOITING its learned Q-values by this point.")
print("Only ~0.67% of actions are random — policy is nearly deterministic.")


---
### Q3 · Pitfalls of the Cowardly Agent
- **Heavy crash penalty + tiny fuel cost** → the agent learns to hover indefinitely
  at altitude, burning fuel forever just to avoid the crash penalty.  
  It "wins" by refusing to descend — a degenerate but locally optimal policy.
- **Huge terminal success reward + no intermediate signals** → the agent receives
  +1000 only on the single terminal step after hundreds of steps of zero signal.
  With a discount factor γ < 1, this reward decays to near-zero for states far from
  the terminal, leaving almost no gradient to learn from. The agent wanders randomly
  and almost never discovers the landing state by chance.

**Solution:** combine a dense step penalty (fuel cost, urgency) with a scaled terminal
reward AND a velocity-scaled crash penalty so every outcome is meaningfully differentiated.

---
### Q4 · The Asymptote of Reward
The theoretical max (+1000) requires a **perfectly fuel-free** descent that lands softly on
the very first try with no engine burns.  
In practice the agent must burn the engine at least a few times to brake (each burn costs
−0.5), and the stochastic wind forces occasional extra burns. The curve therefore approaches
but never reaches the ceiling — it asymptotes slightly below the theoretical maximum.

---
### Q5 · Bridge to the OS Scheduler (Part 6)
| Adrian Probe | OS CPU Scheduler |
|---|---|
| Altitude, Velocity | Queue Length, Avg Burst Time |
| Adrian Storm (TPM) | Stochastic user I/O / network arrivals |
| Thrust ON / OFF | Round Robin vs. Shortest-Job-First |
| Avoid crush / fuel cost | Minimize P90 latency / maximise throughput |

The RL framework is **identical**: a Markov state captures system volatility, a stochastic
transition models unpredictable inputs, and the agent maximises a discounted reward signal
that trades off short-term costs against long-term performance.


---
## Part 1 & 2 — `ProbeEnv`: Deterministic + Stochastic Physics Engine


In [ ]:
class ProbeEnv:
    """
    Physics-based RL environment for the Adrian Descent.
    State : (altitude h [m], velocity v [m/s], wind_idx {0,1,2})
    Action: 0 = Spin-Drive OFF | 1 = Spin-Drive ON
    """
    # ── Physical constants (lore-accurate) ──────────────────────────────────
    MASS     = 1000.0       # kg
    G        = 13.7         # m/s²  (Adrian surface gravity)
    R_ADRIAN = 10_700_000.0 # m     (radius of Adrian)
    K_DRAG   = 2.0          # kg/m  (high-density atmosphere)
    F_THRUST = 25_000.0     # N
    DT       = 0.1          # s

    # ── Wind states: Calm | Gusty | Adrian Gale ─────────────────────────────
    WIND_DRAG_MULT = [1.0, 1.5, 2.5]

    # ── Transition Probability Matrix (rows = current, cols = next) ──────────
    WIND_TRANS = np.array([
        [0.70, 0.25, 0.05],   # Calm  → Calm / Gusty / Gale
        [0.30, 0.50, 0.20],   # Gusty → ...
        [0.10, 0.30, 0.60],   # Gale  → ...
    ])

    INIT_ALT  = 1000.0   # m   drop altitude
    MAX_ALT   = 1200.0   # m   runaway ceiling
    MAX_STEPS = 800      # steps before "hover timeout"
    SAFE_VEL  = -3.0     # m/s threshold for a soft catch

    def __init__(self):
        self.reset()

    # ────────────────────────────────────────────────────────────────────────
    def reset(self):
        """Return probe to drop altitude, zero velocity, calm winds."""
        self.h        = self.INIT_ALT
        self.v        = 0.0
        self.wind_idx = 0
        self.steps    = 0
        return (self.h, self.v, self.wind_idx)

    # ────────────────────────────────────────────────────────────────────────
    def _transition_wind(self):
        """Markov wind state transition."""
        self.wind_idx = int(np.random.choice(3, p=self.WIND_TRANS[self.wind_idx]))

    # ────────────────────────────────────────────────────────────────────────
    def step(self, action):
        """
        Apply forces, Euler-integrate, transition wind, compute reward.
        Returns: (next_state, reward, done)
        """
        # 1. Gravity (altitude-dependent)
        f_grav = -self.MASS * self.G * (1.0 - self.h / self.R_ADRIAN)

        # 2. Thrust (binary)
        f_thrust = self.F_THRUST * action

        # 3. Drag (opposes motion, wind-scaled)
        wind_mult = self.WIND_DRAG_MULT[self.wind_idx]
        f_drag = self.K_DRAG * wind_mult * (self.v ** 2) * np.sign(-self.v)

        # 4. Euler integration
        a      = (f_grav + f_thrust + f_drag) / self.MASS
        self.v = self.v + a * self.DT
        self.h = self.h + self.v * self.DT
        self.steps += 1

        # 5. Stochastic wind transition
        self._transition_wind()

        # ── Reward shaping ─────────────────────────────────────────────────
        reward = -0.5 * action          # continuous fuel cost

        done = False

        if self.h <= 0.0:
            done = True
            if self.v >= self.SAFE_VEL:
                reward += 1000.0        # ✅ Soft Target Catch
            else:
                # 💥 Crushed by pressure — penalty scales with impact speed
                reward += max(-500.0, 2.0 * self.v)

        elif self.h > self.MAX_ALT:
            reward -= 100.0             # 🚀 Runaway probe
            done = True

        elif self.steps >= self.MAX_STEPS:
            reward -= 50.0              # ⏱ Hover timeout
            done = True

        return (self.h, self.v, self.wind_idx), reward, done

    # ────────────────────────────────────────────────────────────────────────
    def __repr__(self):
        wind_names = ['Calm', 'Gusty', 'Adrian Gale']
        return (f"ProbeEnv | h={self.h:.1f}m  v={self.v:.2f}m/s  "
                f"wind={wind_names[self.wind_idx]}  step={self.steps}")


# Quick sanity check
env = ProbeEnv()
s = env.reset()
print(f"Initial state: h={s[0]:.1f}m, v={s[1]:.2f}m/s, wind={s[2]}")
s2, r, d = env.step(0)
print(f"After step(OFF): h={s2[0]:.2f}m, v={s2[1]:.4f}m/s, r={r:.2f}, done={d}")
s3, r2, d2 = env.step(1)
print(f"After step(ON) : h={s3[0]:.2f}m, v={s3[1]:.4f}m/s, r={r2:.2f}, done={d2}")


---
## Part 3 — `ProbeAgent`: Tabular Q-Learning


In [ ]:
class ProbeAgent:
    """
    Tabular Q-Learning agent for the Adrian Descent.
    Discretises the continuous (h, v) state into fixed bins and
    maintains a Q-table of shape [N_ALT × N_VEL × N_WIND × N_ACT].
    """
    N_ALT  = 50
    N_VEL  = 50
    N_WIND = 3
    N_ACT  = 2

    # Bin edges (N-1 edges → N buckets via np.digitize)
    ALT_BINS = np.linspace(0,    1200, N_ALT  - 1)
    VEL_BINS = np.linspace(-100,  20,  N_VEL  - 1)

    def __init__(self,
                 alpha     = 0.10,    # learning rate
                 gamma     = 0.99,    # discount factor
                 epsilon   = 1.00,    # initial exploration rate
                 eps_decay = 0.999,   # per-episode decay
                 eps_min   = 0.01):   # floor
        self.alpha     = alpha
        self.gamma     = gamma
        self.epsilon   = epsilon
        self.eps_decay = eps_decay
        self.eps_min   = eps_min
        # Q-table initialised to zero
        self.Q = np.zeros((self.N_ALT, self.N_VEL, self.N_WIND, self.N_ACT))

    # ────────────────────────────────────────────────────────────────────────
    def discretize_state(self, raw_state):
        """Map continuous (h, v, wind) to integer indices."""
        h, v, wind = raw_state
        i_h = int(np.clip(np.digitize(h, self.ALT_BINS), 0, self.N_ALT  - 1))
        i_v = int(np.clip(np.digitize(v, self.VEL_BINS), 0, self.N_VEL  - 1))
        return (i_h, i_v, int(wind))

    # ────────────────────────────────────────────────────────────────────────
    def choose_action(self, state):
        """ε-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.N_ACT)   # explore
        ih, iv, w = state
        return int(np.argmax(self.Q[ih, iv, w]))   # exploit

    # ────────────────────────────────────────────────────────────────────────
    def learn(self, state, action, reward, next_state, done):
        """
        Bellman update (Temporal Difference):
        Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') – Q(s,a) ]
        """
        ih,  iv,  w  = state
        ihn, ivn, wn = next_state
        q_cur  = self.Q[ih,  iv,  w,  action]
        q_next = 0.0 if done else np.max(self.Q[ihn, ivn, wn])
        td_err = reward + self.gamma * q_next - q_cur
        self.Q[ih, iv, w, action] += self.alpha * td_err

    # ────────────────────────────────────────────────────────────────────────
    def decay_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

    # ────────────────────────────────────────────────────────────────────────
    def __repr__(self):
        qtable_size = self.N_ALT * self.N_VEL * self.N_WIND * self.N_ACT
        return (f"ProbeAgent | Q-table: {qtable_size:,} entries  "
                f"ε={self.epsilon:.4f}  α={self.alpha}  γ={self.gamma}")


agent_test = ProbeAgent()
print(agent_test)
s_test = agent_test.discretize_state((500.0, -20.0, 1))
print(f"Discretized state (h=500, v=-20, wind=Gusty): {s_test}")


---
## Part 5 — Training Loop (18 000 Episodes)


In [ ]:
def moving_avg(data, window=500):
    return np.convolve(data, np.ones(window) / window, mode='valid')


def train(n_episodes=18_000, print_every=2000):
    env   = ProbeEnv()
    agent = ProbeAgent()

    all_rewards  = []
    all_outcomes = []   # 1 = soft catch, 0 = otherwise

    for ep in range(n_episodes):
        raw   = env.reset()
        state = agent.discretize_state(raw)
        total_r = 0.0
        success = False

        while True:
            action               = agent.choose_action(state)
            raw_next, rew, done  = env.step(action)
            next_state           = agent.discretize_state(raw_next)
            agent.learn(state, action, rew, next_state, done)
            state    = next_state
            total_r += rew
            if done:
                if env.h <= 0.0 and env.v >= ProbeEnv.SAFE_VEL:
                    success = True
                break

        agent.decay_epsilon()
        all_rewards.append(total_r)
        all_outcomes.append(1 if success else 0)

        if (ep + 1) % print_every == 0:
            recent_success = 100 * sum(all_outcomes[-print_every:]) / print_every
            recent_reward  = np.mean(all_rewards[-print_every:])
            print(f"  Ep {ep+1:6d}/{n_episodes}  "
                  f"ε={agent.epsilon:.4f}  "
                  f"avg_reward={recent_reward:8.1f}  "
                  f"success={recent_success:.1f}%")

    return agent, all_rewards, all_outcomes


print("🚀 Beginning mission training …")
trained_agent, rewards, outcomes = train(n_episodes=18_000)
print("\n✅ Training complete!")
print(f"   Final ε      : {trained_agent.epsilon:.4f}")
print(f"   Final 500-ep success rate: {100*sum(outcomes[-500:])/500:.1f}%")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle("Project Hail Mary – Adrian Descent: Training Curves", fontsize=14, fontweight='bold')

ma_r = moving_avg(rewards, 500)
ax1.plot(ma_r, color='#4fc3f7', linewidth=1.5, label='Avg Reward')
ax1.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax1.set_title("Episode Reward (500-ep Moving Average)")
ax1.set_xlabel("Episode")
ax1.set_ylabel("Total Reward")
ax1.legend()
ax1.grid(True, alpha=0.3)

ma_s = moving_avg([o * 100 for o in outcomes], 500)
ax2.plot(ma_s, color='#81c784', linewidth=1.5, label='Success Rate %')
ax2.set_title("Landing Success Rate % (500-ep Moving Average)")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Success Rate (%)")
ax2.set_ylim(0, 105)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("learning_curve.png", dpi=150)
plt.show()
print("Plot saved as learning_curve.png")


---
## Part 5 — Evaluation: Greedy Test Episode with ASCII Renderer


In [ ]:
def render_probe_ascii(h, max_h, v, action, wind, step_count, is_jupyter=False):
    """High-framerate pure-ASCII terminal renderer (provided tooling)."""
    if is_jupyter:
        clear_output(wait=True)
    else:
        os.system('clear' if os.name == 'posix' else 'cls')

    term_lines = 40
    if h > 150.0:
        display_max = max_h
        zoom_str = "[ CAMERA: WIDE ANGLE (1000 m) ]"
    else:
        display_max = 150.0
        zoom_str = "[ CAMERA: TARGET APPROACH (150 m) ]"

    pos = int((h / display_max) * term_lines)
    pos = max(0, min(term_lines, pos))

    wind_strs  = ["~  Calm  ~", "  Gusty  ", " Adrian Gale "]
    thrust_str = " [####] ON " if action == 1 else "[      ] OFF"

    print(f" T+{step_count:03d}s | ALT: {h:6.1f}m | VEL: {v:7.2f}m/s | "
          f"THRUST: {thrust_str} | WIND: {wind_strs[wind]}")
    print(f" {zoom_str}")
    print("-" * 75)

    for i in range(term_lines, -1, -1):
        if i == pos:
            if action == 1:
                print("            /\")
                print("            | |")
                print("            /WW\")
                print("            ||  <-- spin-drive")
            else:
                print("            /\")
                print("            | |")
                print("            /--\")
                print("               ")
        else:
            if i % 10 == 0:
                print(f" {int((i/term_lines)*display_max):4d}m +------------------------")
            else:
                print("      |")

    print("==================== [ TAUMOEBA TARGET ] ====================")
    time.sleep(0.04)


def evaluate(agent, render=True, is_jupyter=True):
    """Run one fully greedy episode (ε=0)."""
    env_eval = ProbeEnv()
    saved_eps = agent.epsilon
    agent.epsilon = 0.0         # pure exploitation

    raw   = env_eval.reset()
    state = agent.discretize_state(raw)
    total_r = 0.0
    step  = 0

    print("\n🛸 Beginning greedy descent …\n")

    while True:
        h, v, wind = env_eval.h, env_eval.v, env_eval.wind_idx
        action = agent.choose_action(state)

        if render:
            render_probe_ascii(h, ProbeEnv.INIT_ALT, v, action, wind,
                               step, is_jupyter=is_jupyter)

        raw_next, rew, done = env_eval.step(action)
        next_state = agent.discretize_state(raw_next)
        state  = next_state
        total_r += rew
        step   += 1

        if done:
            if env_eval.h <= 0.0 and env_eval.v >= ProbeEnv.SAFE_VEL:
                print(f"\n✅ SOFT CATCH! v={env_eval.v:.2f} m/s | total_reward={total_r:.1f}")
            elif env_eval.h <= 0.0:
                print(f"\n💥 CRUSHED! v={env_eval.v:.2f} m/s | total_reward={total_r:.1f}")
            else:
                print(f"\n⚠️  Runaway / Timeout | total_reward={total_r:.1f}")
            break

    agent.epsilon = saved_eps
    return total_r


# Run evaluation (set render=False to skip animation if running non-interactively)
eval_reward = evaluate(trained_agent, render=True, is_jupyter=True)


---
##  Bonus — Variable Spin-Drive (3-level Throttle)
Extended action space: **0 = OFF | 1 = 50% Thrust | 2 = 100% Thrust**


In [ ]:
class ProbeEnvBonus(ProbeEnv):
    """Same as ProbeEnv but exposes a 3-level throttle."""
    THRUST_LEVELS = [0.0, 0.5, 1.0]

    def step(self, action):
        throttle = self.THRUST_LEVELS[action]
        # Rebuild Fthrust with partial throttle
        f_grav = -self.MASS * self.G * (1.0 - self.h / self.R_ADRIAN)
        f_thrust = self.F_THRUST * throttle
        wind_mult = self.WIND_DRAG_MULT[self.wind_idx]
        f_drag = self.K_DRAG * wind_mult * (self.v ** 2) * np.sign(-self.v)
        a = (f_grav + f_thrust + f_drag) / self.MASS
        self.v = self.v + a * self.DT
        self.h = self.h + self.v * self.DT
        self.steps += 1
        self._transition_wind()

        reward = -0.3 * throttle        # fuel cost proportional to throttle
        done = False
        if self.h <= 0.0:
            done = True
            reward += 1000.0 if self.v >= self.SAFE_VEL else max(-500.0, 2.0 * self.v)
        elif self.h > self.MAX_ALT:
            reward -= 100.0; done = True
        elif self.steps >= self.MAX_STEPS:
            reward -= 50.0; done = True

        return (self.h, self.v, self.wind_idx), reward, done


class ProbeAgentBonus(ProbeAgent):
    """Q-table extended to 3 actions."""
    N_ACT = 3

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.Q = np.zeros((self.N_ALT, self.N_VEL, self.N_WIND, self.N_ACT))


def train_bonus(n_episodes=18_000):
    env_b   = ProbeEnvBonus()
    agent_b = ProbeAgentBonus()
    all_rewards, all_outcomes = [], []

    for ep in range(n_episodes):
        raw   = env_b.reset()
        state = agent_b.discretize_state(raw)
        total_r, success = 0.0, False
        while True:
            action = agent_b.choose_action(state)
            raw_next, rew, done = env_b.step(action)
            next_state = agent_b.discretize_state(raw_next)
            agent_b.learn(state, action, rew, next_state, done)
            state = next_state; total_r += rew
            if done:
                if env_b.h <= 0.0 and env_b.v >= ProbeEnv.SAFE_VEL:
                    success = True
                break
        agent_b.decay_epsilon()
        all_rewards.append(total_r)
        all_outcomes.append(1 if success else 0)

    return agent_b, all_rewards, all_outcomes


bonus_n_act = ProbeAgentBonus.N_ALT * ProbeAgentBonus.N_VEL * ProbeAgentBonus.N_WIND * ProbeAgentBonus.N_ACT
binary_n_act = ProbeAgent.N_ALT * ProbeAgent.N_VEL * ProbeAgent.N_WIND * ProbeAgent.N_ACT
print(f"Binary Q-table size  : {binary_n_act:,} entries")
print(f"3-action Q-table size: {bonus_n_act:,} entries  (+{bonus_n_act - binary_n_act:,} extra)")

print("\nTraining bonus agent …")
bonus_agent, bonus_rewards, bonus_outcomes = train_bonus(18_000)
print(f"Bonus final success rate: {100*sum(bonus_outcomes[-500:])/500:.1f}%")

# Compare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Bonus: Binary vs 3-Level Throttle Comparison", fontsize=13, fontweight='bold')

for ax, rews, outs, label, color in zip(
        axes,
        [rewards,       bonus_rewards],
        [outcomes,      bonus_outcomes],
        ['Binary (ON/OFF)', '3-Level Throttle'],
        ['#4fc3f7', '#ffb74d']):
    ma = moving_avg([o*100 for o in outs], 500)
    ax.plot(ma, color=color, linewidth=1.5)
    ax.set_title(f"{label}")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Success Rate (%)")
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("bonus_comparison.png", dpi=150)
plt.show()
print("Bonus comparison saved as bonus_comparison.png")


---
### Bonus Analysis

**Does a finer-grained action space help?**  
Yes — the 50% throttle level allows the probe to fine-tune its deceleration near the surface,
reducing overshoot and achieving softer landings compared to the bang-bang binary policy.

**Q-table size change:**  
The table grows from **15 000 → 22 500 entries** (50×50×3×3 vs ×2) — a 50% increase.
This modestly increases convergence time since more entries need to be visited and updated,
but it is far less severe than doubling the discretisation resolution would be.
